# 🎯 Tạo Dataset ReID cho OSNet - SoccerTrack
**Mục tiêu**: Từ video + annotation CSV → tạo thư mục `football_reid` theo đúng format cho fine-tune OSNet (CE + Triplet loss)

**Cấu trúc output**:
football_reid/  
├── train/  
│   ├── video001_player_01/  
│   ├── video001_player_02/  
│   └── ...  
├── test/  
│   ├── query/  
│   └── gallery/  

- **Train**: 3 video đầu tiên
- **Subsample**: every 5 frames → ~5 FPS (từ 25 FPS gốc)
- **Test**: 6 video cuối (tự động chia query / gallery)

In [1]:
import cv2
import pandas as pd
import os
import numpy as np
from collections import defaultdict
from tqdm.notebook import tqdm  
import warnings
warnings.filterwarnings("ignore")

In [7]:
# ========================== CELL 2: CONFIG ==========================
INPUT_DIR       = r"Data\wide_view\videos"
ANNOTATION_DIR  = r"Data\wide_view\annotations"  

REID_OUTPUT     = "Data/ReID"

# Tự động lấy danh sách video
ALL_VIDEOS = sorted([f for f in os.listdir(INPUT_DIR) 
                     if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))])

TRAIN_VIDEOS = ALL_VIDEOS[:3]      
TEST_VIDEOS  = ALL_VIDEOS[-6:-4]     

print(f"Tổng số video tìm thấy: {len(ALL_VIDEOS)}")
print(f"Train videos : {len(TRAIN_VIDEOS)} → {TRAIN_VIDEOS}")
print(f"Test videos  : {len(TEST_VIDEOS)} → {TEST_VIDEOS}")

Tổng số video tìm thấy: 66
Train videos : 3 → ['F_20200220_1_0000_0030.mp4', 'F_20200220_1_0030_0060.mp4', 'F_20200220_1_0060_0090.mp4']
Test videos  : 2 → ['F_20220220_1_1800_1830.mp4', 'F_20220220_1_1830_1860.mp4']


In [4]:
def build_object_list(raw):
    """Hàm parse annotation CSV của bạn"""
    n_cols = raw.iloc[0].tolist()
    objects = []
    col = 1
    while col + 3 < len(n_cols):
        team = str(n_cols[col]).strip().upper()
        if team != "BALL":
            objects.append((1, col, col + 1, col + 2, col + 3))
        col += 4
    return objects

In [5]:
def process_video(video_name: str, is_test: bool = False, query_ratio: float = 0.3, frame_step: int = 5):
    """
    frame_step = 5 → subsample every 5 frames ≈ 5 FPS (từ 25 FPS gốc)
    """
    video_path = os.path.join(INPUT_DIR, video_name)
    csv_path   = os.path.join(ANNOTATION_DIR, os.path.splitext(video_name)[0] + ".csv")

    if not os.path.exists(csv_path):
        print(f"⚠️ Không tìm thấy annotation: {csv_path}")
        return

    raw = pd.read_csv(csv_path, header=None)
    objects = build_object_list(raw)
    num_players = len(objects)

    # Thu thập bbox
    player_data = defaultdict(list)
    data_rows = raw.iloc[4:]

    for _, row in tqdm(data_rows.iterrows(), total=len(data_rows), desc=f"📊 {video_name}"):
        vals = row.tolist()
        try:
            frame_id = int(float(str(vals[0]).strip()))
        except:
            continue

        for p_idx, (_, h_col, l_col, t_col, w_col) in enumerate(objects):
            try:
                h = float(vals[h_col])
                left = float(vals[l_col])
                top = float(vals[t_col])
                w = float(vals[w_col])
            except:
                continue
            if any(np.isnan(v) for v in [h, left, top, w]) or w <= 0 or h <= 0:
                continue
            player_data[p_idx].append((frame_id, left, top, w, h))

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Không mở được video: {video_name}")
        return

    video_stem = os.path.splitext(video_name)[0]

    for p_idx in range(num_players):
        player_name = f"{video_stem}_player_{p_idx+1:02d}"
        track = sorted(player_data[p_idx], key=lambda x: x[0])
        
        if not track:
            continue

        # === SUBSAMPLE THEO 5 FPS ===
        subsampled_track = track[::frame_step]
        print(f"   {player_name}: {len(track)} → {len(subsampled_track)} frames (step={frame_step})")

        # ===================== TRAIN =====================
        if not is_test:
            save_dir = os.path.join(REID_OUTPUT, "train", player_name)
            os.makedirs(save_dir, exist_ok=True)
            crop_list = subsampled_track

            for fid, left, top, w, h in crop_list:
                cap.set(cv2.CAP_PROP_POS_FRAMES, fid)
                ret, frame = cap.read()
                if not ret: continue
                x1 = max(0, int(left))
                y1 = max(0, int(top))
                x2 = min(frame.shape[1], int(left + w))
                y2 = min(frame.shape[0], int(top + h))
                if x2 - x1 < 10 or y2 - y1 < 10: continue
                crop = frame[y1:y2, x1:x2]
                cv2.imwrite(os.path.join(save_dir, f"f{fid:04d}.jpg"), crop)

        # ===================== TEST =====================
        else:
            q_split = int(len(subsampled_track) * query_ratio)
            query_list = subsampled_track[:q_split]
            gallery_list = subsampled_track[q_split:]

            # Query
            q_dir = os.path.join(REID_OUTPUT, "test", "query", player_name)
            os.makedirs(q_dir, exist_ok=True)
            for fid, left, top, w, h in query_list:
                cap.set(cv2.CAP_PROP_POS_FRAMES, fid)
                ret, frame = cap.read()
                if not ret: continue
                x1 = max(0, int(left))
                y1 = max(0, int(top))
                x2 = min(frame.shape[1], int(left + w))
                y2 = min(frame.shape[0], int(top + h))
                if x2 - x1 < 10 or y2 - y1 < 10: continue
                crop = frame[y1:y2, x1:x2]
                cv2.imwrite(os.path.join(q_dir, f"f{fid:04d}.jpg"), crop)

            # Gallery
            g_dir = os.path.join(REID_OUTPUT, "test", "gallery", player_name)
            os.makedirs(g_dir, exist_ok=True)
            for fid, left, top, w, h in gallery_list:
                cap.set(cv2.CAP_PROP_POS_FRAMES, fid)
                ret, frame = cap.read()
                if not ret: continue
                x1 = max(0, int(left))
                y1 = max(0, int(top))
                x2 = min(frame.shape[1], int(left + w))
                y2 = min(frame.shape[0], int(top + h))
                if x2 - x1 < 10 or y2 - y1 < 10: continue
                crop = frame[y1:y2, x1:x2]
                cv2.imwrite(os.path.join(g_dir, f"f{fid:04d}.jpg"), crop)

    cap.release()
    print(f"✅ Hoàn thành {video_name} ({num_players} players)")

In [ ]:
import os
os.makedirs(REID_OUTPUT, exist_ok=True)
os.makedirs(os.path.join(REID_OUTPUT, "train"), exist_ok=True)
os.makedirs(os.path.join(REID_OUTPUT, "test", "query"), exist_ok=True)
os.makedirs(os.path.join(REID_OUTPUT, "test", "gallery"), exist_ok=True)

print("🔄 Bắt đầu xử lý 3 video TRAIN (subsample 5 FPS)...")
for vid in TRAIN_VIDEOS:
    process_video(vid, is_test=False, frame_step=5)

🔄 Bắt đầu xử lý 3 video TRAIN (subsample 5 FPS)...


📊 F_20200220_1_0000_0030.mp4:   0%|          | 0/750 [00:00<?, ?it/s]

   F_20200220_1_0000_0030_player_01: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_02: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_03: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_04: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_05: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_06: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_07: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_08: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_09: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_10: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_11: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_12: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_13: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_14: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_15: 750 → 150 frames (step=5)
   F_20200220_1_0000_0030_player_16: 750 → 150 frames (

📊 F_20200220_1_0030_0060.mp4:   0%|          | 0/750 [00:00<?, ?it/s]

   F_20200220_1_0030_0060_player_01: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_02: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_03: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_04: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_05: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_06: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_07: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_08: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_09: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_10: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_11: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_12: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_13: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_14: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_15: 750 → 150 frames (step=5)
   F_20200220_1_0030_0060_player_16: 750 → 150 frames (

📊 F_20200220_1_0060_0090.mp4:   0%|          | 0/750 [00:00<?, ?it/s]

   F_20200220_1_0060_0090_player_01: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_02: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_03: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_04: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_05: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_06: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_07: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_08: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_09: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_10: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_11: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_12: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_13: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_14: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_15: 750 → 150 frames (step=5)
   F_20200220_1_0060_0090_player_16: 750 → 150 frames (

📊 F_20220220_1_1800_1830.mp4:   0%|          | 0/750 [00:00<?, ?it/s]

   F_20220220_1_1800_1830_player_01: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_02: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_03: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_04: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_05: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_06: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_07: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_08: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_09: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_10: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_11: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_12: 750 → 150 frames (step=5)


KeyboardInterrupt: 

In [9]:
print("\n🔄 Bắt đầu xử lý 2 video TEST (subsample 5 FPS + chia query/gallery)...")
for vid in TEST_VIDEOS:
    process_video(vid, is_test=True, query_ratio=0.3, frame_step=5)

print("\n HOÀN TẤT!")


🔄 Bắt đầu xử lý 2 video TEST (subsample 5 FPS + chia query/gallery)...


📊 F_20220220_1_1800_1830.mp4:   0%|          | 0/750 [00:00<?, ?it/s]

   F_20220220_1_1800_1830_player_01: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_02: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_03: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_04: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_05: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_06: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_07: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_08: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_09: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_10: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_11: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_12: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_13: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_14: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_15: 750 → 150 frames (step=5)
   F_20220220_1_1800_1830_player_16: 750 → 150 frames (

📊 F_20220220_1_1830_1860.mp4:   0%|          | 0/750 [00:00<?, ?it/s]

   F_20220220_1_1830_1860_player_01: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_02: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_03: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_04: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_05: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_06: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_07: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_08: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_09: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_10: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_11: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_12: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_13: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_14: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_15: 750 → 150 frames (step=5)
   F_20220220_1_1830_1860_player_16: 750 → 150 frames (